Se formalizan los resultados en el notebook edm_ncuerpos_Jaider

In [ ]:
import pymcel as pc
from scipy.integrate import odeint
import random
import math
import numpy as np
import matplotlib.pyplot as plt
from celluloid import Camera
import plotly.graph_objects as go
from IPython.display import HTML, display

In [120]:
G = 1

In [121]:
#función para crear el sistema de masa central (muy masiva) y N cuerpos de baja masa orbitando alrededor de ella
def crear_sistema(m_central,num_stars,radius_min,radius_max,sentido_rotacion):
    sistema = [dict(m=m_central, r=[0, 0, 0], v=[0, 0, 0])]  # masa central en el origen
    for _ in range(num_stars):
        rho = random.uniform(radius_min, radius_max)
        theta = random.uniform(0, 2 * math.pi)
        x = rho * math.cos(theta)
        y = rho * math.sin(theta)

        decision = random.randint(0, 99)

        if decision <= 50:
         v_mag = np.sqrt(2*m_central / rho)
        else:
         v_mag = (np.sqrt(2*m_central / rho) - random.uniform(0.90, 1.00))

        vx = -sentido_rotacion * v_mag * np.sin(theta)
        vy = -sentido_rotacion * v_mag * np.cos(theta)

        sistema.append(dict(m=1e-10, r=[x, y, 0], v=[vx, vy, 0]))
        N, mus, Y0 = pc.sistema_a_Y(sistema)
    
    return sistema, N, mus, Y0

In [122]:
#funcion para resolver el sistema con odeint (edm_ncuerpos) y con un evento (para detener
#la integración si alguna partícula se acerca demasiado al cuerpo central)
def solucion_edm(sistema,N,mus,Y0,ts):
    sistema = sistema
    r_stop = 0.5  # distancia de parada respecto al cuerpo 1 (indice 0)

    Y_curr = Y0.copy()
    sol_parts = [Y_curr.copy()]
    t_parts = [ts[0]]

    stopped = False
    stop_particle = None

    for k in range(len(ts) - 1):
     t_pair = ts[k:k + 2]
     seg = odeint(pc.edm_ncuerpos, Y_curr, t_pair, args=(N, mus))
     Y_next = seg[-1]

     # Extrae posiciones en el ultimo instante para revisar cercania al cuerpo central.
     rs_step, _ = pc.solucion_a_estado(np.array([Y_next]), N, 1)
     if N > 1:
        dists = np.linalg.norm(rs_step[1:, 0, :] - rs_step[0, 0, :], axis=1)
        min_dist = np.min(dists)
        if min_dist <= r_stop:
            stopped = True
            stop_particle = int(np.argmin(dists)) + 1
            t_parts.append(t_pair[-1])
            sol_parts.append(Y_next.copy())
            break

     t_parts.append(t_pair[-1])
     sol_parts.append(Y_next.copy())
     Y_curr = Y_next
    
    ts_event = np.array(t_parts)
    solucion_ejemplo = np.vstack(sol_parts)
    rs, vs = pc.solucion_a_estado(solucion_ejemplo, N, len(ts_event))

    if stopped:
     print(f'Integracion detenida en t={ts_event[-1]:.3f}: particula {stop_particle + 1} se acerco a {r_stop}.')
    else:
     print('No se activo el evento de cercania; se completo toda la integracion.')
    
    return rs, vs, ts_event, sistema, stopped, stop_particle, solucion_ejemplo


In [123]:
#funcion para animar el sistema con matplotlib y celluloid
def animar_sistema_2d(rs, ts):
 fig, ax = plt.subplots(figsize=(6, 6))
 camera = Camera(fig)

 # Fondo del grafico (una sola vez).
 ax.set_aspect('equal', adjustable='box')
 ax.grid(alpha=0.3)

 # Menos cuadros => render mas rapido.
 step = 10
 n_particles = rs.shape[0]
 colors = plt.cm.viridis(np.linspace(0, 1, n_particles))

 for i in range(0, len(ts_event), step):
    for p in range(n_particles):
        # Resaltar el cuerpo central (indice 0).
        lw = 1.2 if p == 0 else 0.8
        alpha = 0.35 if p == 0 else 0.2
        ms = 7 if p == 0 else 3

        ax.plot(rs[p, :, 0], rs[p, :, 1], '-', color=colors[p], lw=lw, alpha=alpha)
        ax.plot(rs[p, i, 0], rs[p, i, 1], 'o', color=colors[p], ms=ms)

    camera.snap()

 anim = camera.animate(interval=60, blit=False)
 plt.close(fig)

 return HTML(anim.to_jshtml())

In [124]:
#funcion para cambiar el sistema si el evento se activa (particula se acerca al cuerpo central)
def cambiar_sistema_por_evento(sistema, stopped, stop_particle):
 if stopped and stop_particle is not None:
    # stop_particle ya esta en el mismo indice usado por 'sistema' (0 = central).
    idx_remove = stop_particle

    removed = sistema[idx_remove]
    sistema = [p for j, p in enumerate(sistema) if j != idx_remove]
    num_stars = len(sistema) - 1

    print(f"Particula removida: indice {idx_remove} (particula {idx_remove + 1} en la grafica).")
    print(f"Nueva cantidad total de cuerpos: {len(sistema)}")

    # Recalcula solucion con el sistema filtrado.
    ts = np.linspace(0, 100, 1000)
    rs, vs, rps, vps, constantes = pc.ncuerpos_solucion(sistema, ts)
 else:
    print("No se removio ninguna particula: no hay evento registrado.")

 return sistema

In [125]:
def redefinicion(sistema, stopped, stop_particle, solucion_ejemplo):
  if 'sistema' not in globals():
    sistema = sistema.copy()

  if stopped and stop_particle is not None:
    # Construye Y0 del sistema filtrado usando el ultimo estado integrado por odeint.
    Y_last = solucion_ejemplo[-1]
    N_old = len(Y_last) // 6

    r_last = Y_last[:3 * N_old].reshape(N_old, 3)
    v_last = Y_last[3 * N_old:].reshape(N_old, 3)

    keep = [j for j in range(N_old) if j != stop_particle]
    r_new = r_last[keep]
    v_new = v_last[keep]

    Y0 = np.concatenate([r_new.reshape(-1), v_new.reshape(-1)])
    N = len(keep)
    mus = np.array([c['m'] for c in sistema], dtype=float)
  else:
    # Si no hubo evento, usa las condiciones iniciales del sistema actual.
    N, mus, Y0 = pc.sistema_a_Y(sistema)

  print('N nuevo:', N)
  print('len(Y0):', len(Y0))

  return sistema, N, mus, Y0

In [127]:
sistema, N, mus, Y0 = crear_sistema(m_central=50, num_stars=10, radius_min=10, radius_max=20, sentido_rotacion=1)
ts = np.linspace(0, 100, 1000)
max_iter = 20

for iteracion in range(1, max_iter + 1):
    print(f"\nIteracion {iteracion} - cuerpos actuales: {len(sistema)}")

    rs, vs, ts_event, sistema, stopped, stop_particle, solucion_ejemplo = solucion_edm(sistema, N, mus, Y0, ts=ts)

    if not stopped:
        print("Proceso finalizado: no se activo el evento.")
        break

    sistema = cambiar_sistema_por_evento(sistema, stopped, stop_particle)
    sistema, N, mus, Y0 = redefinicion(sistema, stopped, stop_particle, solucion_ejemplo)

    display(animar_sistema_2d(rs, ts_event))

    if len(sistema) <= 1:
        print("Solo queda el cuerpo central; no hay mas particulas para integrar.")
        break
else:
    print("Se alcanzo el maximo de iteraciones sin converger.")

print(f"\nSistema final con {len(sistema)} cuerpos.")


Iteracion 1 - cuerpos actuales: 11
Integracion detenida en t=27.127: particula 6 se acerco a 0.5.
Particula removida: indice 5 (particula 6 en la grafica).
Nueva cantidad total de cuerpos: 10
N nuevo: 10
len(Y0): 60



Iteracion 2 - cuerpos actuales: 10


C:\Users\jaide\AppData\Local\Temp\ipykernel_15832\2182242257.py:16: ODEintWarning: Excess work done on this call (perhaps wrong Dfun type). Run with full_output = 1 to get quantitative information.
  seg = odeint(pc.edm_ncuerpos, Y_curr, t_pair, args=(N, mus))


Integracion detenida en t=15.616: particula 10 se acerco a 0.5.
Particula removida: indice 9 (particula 10 en la grafica).
Nueva cantidad total de cuerpos: 9
N nuevo: 9
len(Y0): 54



Iteracion 3 - cuerpos actuales: 9
No se activo el evento de cercania; se completo toda la integracion.
Proceso finalizado: no se activo el evento.

Sistema final con 9 cuerpos.
